In [ ]:
# ============================================
# 🌿 Plant Disease CNN — No Kaggle Required
# One-cell Google Colab pipeline
# ============================================

!pip -q install tensorflow gdown

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os, zipfile, gdown

print("TensorFlow:", tf.__version__)

# -------------------------------------------------
# 1) Download PlantVillage dataset (public mirror)
# -------------------------------------------------
# 1.2GB dataset hosted publicly (no login needed)
url = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"
gdown.download(url, "plantvillage.zip", quiet=False)

!unzip -q plantvillage.zip
DATASET_DIR = "PlantVillage-Dataset-master/raw/color"

# -------------------------------------------------
# 2) Load dataset
# -------------------------------------------------
IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Number of classes:", NUM_CLASSES)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# -------------------------------------------------
# 3) Data Augmentation
# -------------------------------------------------
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

# -------------------------------------------------
# 4) Transfer Learning Model (High Accuracy)
# -------------------------------------------------
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(224,224,3)),
    data_aug,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# -------------------------------------------------
# 5) Train Model
# -------------------------------------------------
EPOCHS = 5
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# -------------------------------------------------
# 6) Plot Results
# -------------------------------------------------
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Accuracy"); plt.legend(["Train","Val"])

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Loss"); plt.legend(["Train","Val"])
plt.show()

# -------------------------------------------------
# 7) Test Prediction Example
# -------------------------------------------------
sample_batch = next(iter(val_ds))
img = sample_batch[0][0]
true_label = class_names[np.argmax(sample_batch[1][0])]

pred = model.predict(tf.expand_dims(img,0))
pred_label = class_names[np.argmax(pred)]

plt.imshow(img.numpy())
plt.title(f"Predicted: {pred_label} | Actual: {true_label}")
plt.axis("off")
plt.show()